# EDA · Clasificación de casos legales
**CP2 de Cami — notebook preparado, resultados pendientes de datos locales.**

Analiza únicamente train y validación. No abre test, no descarga datos y no
modifica archivos de datos, modelos ni configuración. Los gráficos son agregados.
Si falta una entrada, la sección se marca pendiente; eso no equivale a pasar la comprobación.

La revisión del piloto está desactivada por defecto. No se muestra texto de casos,
nombres ni palabras extraídas del piloto en las salidas. Antes de compartir este
notebook, borrar todas las salidas y revisar que no se hayan añadido celdas con texto real.


In [1]:
from pathlib import Path
from collections import Counter
import json
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.feature_extraction.text import TfidfVectorizer

override = os.environ.get("EDA_ROOT")
candidates = [Path(override)] if override else [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "configs/config.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abre el notebook desde la raíz del repo o desde notebooks/.")
cfg = yaml.safe_load((ROOT / "configs/config.yaml").read_text(encoding="utf-8"))
AREAS = cfg["areas"]
PROCESSED = ROOT / cfg["paths"]["processed"]
EXTERNAL = ROOT / cfg["paths"]["external"]
INTERIM = ROOT / cfg["paths"]["interim"]
INCLUIR_PILOTO_REVISADO = False  # Activar solo tras CP1 y con autorización aplicable.
SEED = cfg["seed"]
estado = []

def marcar(seccion, detalle):
    estado.append({"seccion": seccion, "estado": detalle})
    print(f"{seccion}: {detalle}")

def normalizar(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", text).strip()

def sin_tildes(text):
    return "".join(c for c in unicodedata.normalize("NFD", normalizar(text))
                   if unicodedata.category(c) != "Mn")

def leer_split_permitido(nombre):
    if nombre not in {"train", "val"}:
        raise PermissionError("Este notebook permite únicamente train y val.")
    path = PROCESSED / f"{nombre}.csv"
    if not path.is_file():
        marcar(nombre, "PENDIENTE: archivo no disponible")
        return None
    df = pd.read_csv(path, encoding="utf-8", dtype={"case_id": str})
    required = {"case_id", "texto", "area_legal", "canal", "fecha", "fuente_texto"}
    if not required <= set(df):
        raise ValueError(f"{nombre}: faltan columnas del contrato")
    if df.empty or df[["case_id", "texto", "area_legal"]].isna().any().any():
        raise ValueError(f"{nombre}: datos vacíos o campos obligatorios nulos")
    if not df.area_legal.isin(AREAS).all() or not df.texto.astype(str).str.strip().ne("").all():
        raise ValueError(f"{nombre}: área desconocida o texto vacío")
    if not df.canal.isin(cfg["canales"]).all() or not df.fuente_texto.isin(["nativo", "ocr"]).all():
        raise ValueError(f"{nombre}: canal o fuente de texto inválidos")
    df = df.copy()
    df["split_eda"] = nombre
    marcar(nombre, f"CARGADO: {len(df)} registros")
    return df

train = leer_split_permitido("train")
val = leer_split_permitido("val")
parts = [d for d in (train, val) if d is not None]
datos = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
if train is not None and val is not None:
    if train.case_id.duplicated().any() or val.case_id.duplicated().any():
        raise ValueError("Hay IDs repetidos dentro de una partición; resolver antes de continuar.")
    if set(train.case_id) & set(val.case_id):
        raise ValueError("Hay IDs compartidos entre train y val; resolver antes de continuar.")


train: PENDIENTE: archivo no disponible
val: PENDIENTE: archivo no disponible


## 1. Balance de clases y procedencia
Una muestra equilibrada por diseño no representa necesariamente la frecuencia
real del bufete. Se reportan conteos y proporciones sin ejemplos textuales.


In [2]:
if not datos.empty:
    balance = pd.crosstab(datos.area_legal, datos.split_eda).reindex(AREAS, fill_value=0)
    display(balance)
    display(balance.div(balance.sum(axis=0), axis=1).round(4))
    ax = balance.plot.bar(figsize=(9, 4), rot=0, title="Casos por área y partición")
    ax.set(xlabel="Área legal", ylabel="Casos")
    plt.tight_layout(); plt.show()
    display(pd.crosstab([datos.split_eda, datos.canal], datos.fuente_texto))
else:
    marcar("Balance", "PENDIENTE: faltan datos")


Balance: PENDIENTE: faltan datos


## 2. Composición de `familia` y `otros`
Se leen de la fuente únicamente ID y metadatos de etiqueta, nunca su texto.
Solo se analizan IDs presentes en train/val. No se reconstruyen particiones.
La etiqueta `familia` puede proceder de reglas léxicas sobre CIVIL; un resultado
alto en esa clase no demuestra que se generalice a consultas del bufete.


In [3]:
metadata_path = EXTERNAL / "saij_por_area.csv"
if datos.empty:
    marcar("Etiquetas", "PENDIENTE: faltan train/val")
elif not metadata_path.is_file():
    marcar("Etiquetas", "PENDIENTE: falta saij_por_area.csv")
else:
    wanted = {"case_id", "materia_original", "etiqueta"}
    header = set(pd.read_csv(metadata_path, nrows=0).columns)
    if not wanted <= header:
        marcar("Etiquetas", "PENDIENTE: faltan columnas de procedencia")
    else:
        allowed_ids = set(datos.case_id)
        chunks = []
        for chunk in pd.read_csv(metadata_path, usecols=sorted(wanted), dtype=str, chunksize=10000):
            chunks.append(chunk[chunk.case_id.isin(allowed_ids)])
        meta = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=sorted(wanted))
        if meta.case_id.duplicated().any():
            raise ValueError("Metadatos con IDs repetidos: revisar la fuente.")
        enriched = datos[["case_id", "area_legal", "split_eda"]].merge(meta, on="case_id", how="left", validate="one_to_one")
        print("Casos sin metadatos completos:", int(enriched[list(wanted - {"case_id"})].isna().any(axis=1).sum()))
        display(pd.crosstab(enriched.area_legal, enriched.etiqueta).reindex(AREAS, fill_value=0))
        for area in ("familia", "otros"):
            subset = enriched[enriched.area_legal == area]
            print("Materia de origen —", area)
            display(subset.groupby(["split_eda", "materia_original"], dropna=False).size().rename("casos").to_frame())
        marcar("Etiquetas", "CALCULADO: revisar cobertura y procedencia")


Etiquetas: PENDIENTE: faltan train/val


## 3. Longitud y piloto local opcional
La longitud se mide por palabras separadas por espacios, no por tokens BETO.
El piloto solo se carga con la opción explícita y se limita a filas marcadas
`revision_humana = si`. La muestra resultante puede ser menor que los 47
fragmentos mencionados en las tareas. Los fragmentos no son casos independientes.


In [4]:
piloto = None
if INCLUIR_PILOTO_REVISADO:
    pilot_path = INTERIM / "piloto/fragmentos.csv"
    if not pilot_path.is_file():
        marcar("Piloto", "PENDIENTE: archivo no disponible")
    else:
        raw_pilot = pd.read_csv(pilot_path, encoding="utf-8")
        if not {"texto", "revision_humana"} <= set(raw_pilot):
            raise ValueError("El piloto no tiene texto y estado de revisión.")
        mask = raw_pilot.revision_humana.astype(str).str.strip().str.casefold().eq("si")
        piloto = raw_pilot.loc[mask & raw_pilot.texto.notna()].copy()
        print("Filas revisadas utilizables:", len(piloto), "/", len(raw_pilot))
        if piloto.empty:
            piloto = None
            marcar("Piloto", "PENDIENTE: no hay fragmentos revisados utilizables")
        del raw_pilot
else:
    marcar("Piloto", "DESACTIVADO: requiere revisión humana local")

longitudes = []
for name, frame in (("train", train), ("val", val), ("piloto_revisado", piloto)):
    if frame is not None:
        lengths = frame.texto.str.split().str.len()
        longitudes.append(dict(fuente=name, n=len(lengths), minimo=lengths.min(),
                               p10=lengths.quantile(.1), mediana=lengths.median(),
                               p90=lengths.quantile(.9), maximo=lengths.max()))
if longitudes:
    display(pd.DataFrame(longitudes).set_index("fuente").round(1))
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, frame in (("train", train), ("val", val), ("piloto_revisado", piloto)):
        if frame is not None:
            ax.hist(frame.texto.str.split().str.len(), bins=30, alpha=.45, density=True, label=name)
    ax.set(xlabel="Palabras por texto", ylabel="Densidad", title="Distribución de longitud")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    marcar("Longitudes", "PENDIENTE: faltan datos")


Piloto: DESACTIVADO: requiere revisión humana local
Longitudes: PENDIENTE: faltan datos


## 4. Duplicados y casi duplicados
Se distinguen coincidencias exactas de coincidencias tras normalizar espacios y
mayúsculas. Para casi duplicados se ajusta TF-IDF **solo con train** y se calcula
el máximo coseno de cada texto de validación contra train, por bloques.
El umbral usa el metadato existente si está disponible, o 0,90 como referencia.
Los descartes ya realizados se toman de `splits_info.json`; no se rehacen los splits.


In [5]:
info_path = PROCESSED / "splits_info.json"
near_threshold = .90
if info_path.is_file():
    info = json.loads(info_path.read_text(encoding="utf-8"))
    near_threshold = float(info.get("umbral_casi_duplicado", .90))
    print("Umbral documentado de casi duplicados:", near_threshold)
    print("Descartes documentados de validación:", info.get("descartados_por_casi_duplicado", {}).get("val", "sin dato"))
    print("División temporal configurada:", cfg["split"]["temporal"], "· semilla:", SEED)
else:
    marcar("Metadatos de split", "PENDIENTE: falta splits_info.json; referencia coseno=0,90")

if not datos.empty:
    dup_rows = []
    for name, frame in (("train", train), ("val", val)):
        if frame is not None:
            dup_rows.append(dict(split=name, duplicados_exactos=int(frame.texto.duplicated().sum()),
                                 duplicados_normalizados=int(frame.texto.map(normalizar).duplicated().sum())))
    display(pd.DataFrame(dup_rows))
if train is not None and val is not None:
    normalized_train = set(train.texto.map(normalizar))
    print("Textos val coincidentes con train tras normalizar:", int(val.texto.map(normalizar).isin(normalized_train).sum()))
    vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=2)
    try:
        x_train = vectorizer.fit_transform(train.texto)
        x_val = vectorizer.transform(val.texto)
    except ValueError:
        marcar("Casi duplicados", "PENDIENTE: vocabulario insuficiente para TF-IDF con min_df=2")
    else:
        maxima = np.zeros(len(val))
        for start in range(0, len(val), 250):
            maxima[start:start + 250] = (x_val[start:start + 250] @ x_train.T).max(axis=1).toarray().ravel()
        print("Val con coseno >= umbral:", int((maxima >= near_threshold).sum()), "/", len(val))
        display(pd.Series(maxima).describe(percentiles=[.5, .9, .95, .99]).to_frame("coseno_maximo"))
        marcar("Casi duplicados", "CALCULADO: no equivale a revisar fugas por expediente o fuente")
else:
    marcar("Duplicados cruzados", "PENDIENTE: se necesitan train y val")


Metadatos de split: PENDIENTE: falta splits_info.json; referencia coseno=0,90
Duplicados cruzados: PENDIENTE: se necesitan train y val


## 5. Señales superficiales y posibles atajos
Se mide la presencia de tribunal/cámara/sala y términos del área. Son indicadores
para investigar; una mención no prueba fuga por sí sola y su ausencia no descarta
otras fugas. No se abren ni se muestran ejemplos de casos.


In [6]:
AREA_TERMS = {
    "laboral": r"\b(laboral|trabajo)\b",
    "civil": r"\bcivil\b",
    "mercantil": r"\b(mercantil|comercial)\b",
    "penal": r"\bpenal\b",
    "familia": r"\b(familia|divorcio|filiacion|adopcion)\b",
    "otros": r"\b(administrativo|constitucional|seguridad social)\b",
}
if not datos.empty:
    text_norm = datos.texto.map(sin_tildes)
    features = datos[["split_eda", "area_legal"]].copy()
    for term in ("tribunal", "camara", "sala"):
        features["menciona_" + term] = text_norm.str.contains(r"\b" + term + r"\b", regex=True)
    features["menciona_terminos_area"] = [bool(re.search(AREA_TERMS[a], text))
                                             for a, text in zip(datos.area_legal, text_norm)]
    display((features.groupby(["split_eda", "area_legal"]).mean() * 100).round(1))
    print("Porcentajes de textos con mención; patrones exploratorios, no detección exhaustiva.")
else:
    marcar("Atajos", "PENDIENTE: faltan textos")


Atajos: PENDIENTE: faltan textos


## 6. Brecha de vocabulario
Se excluyen marcadores de anonimización y palabras funcionales. Solo se publican
conteos agregados de vocabulario y una lista fija de términos jurídicos.
Las palabras exclusivas del piloto quedan en una variable privada para inspección
local de Cami; no se imprimen ni se exportan. No añadir esa variable como salida.
La comparación SAIJ-train frente a un piloto solo laboral confunde diferencias
de país, canal, tipo de documento y clase; no mide únicamente adaptación de dominio.


In [7]:
STOP = set("de la el los las un una y o a en por para con del al que se su sus es no como lo mas esta este ha son sobre entre sin".split())

def frecuencias_privadas(texts):
    counts = Counter()
    for text in texts:
        text = re.sub(r"\[[^\]]+\]", " ", str(text))
        counts.update(w for w in re.findall(r"[a-z]{3,}", sin_tildes(text)) if w not in STOP)
    return counts

if train is not None and piloto is not None:
    freq_saij = frecuencias_privadas(train.texto)
    freq_piloto = frecuencias_privadas(piloto.texto)
    compartidas = set(freq_saij) & set(freq_piloto)
    print("Tipos léxicos SAIJ-train:", len(freq_saij))
    print("Tipos léxicos piloto revisado:", len(freq_piloto))
    print("Tipos compartidos:", len(compartidas))
    print("Tipos exclusivos de piloto:", len(set(freq_piloto) - set(freq_saij)))
    # PRIVADO: solo para inspección local, nunca imprimir ni exportar automáticamente.
    terminos_piloto_exclusivos_privados = [(w, n) for w, n in freq_piloto.most_common() if w not in freq_saij]
    terminos_saij_exclusivos_privados = [(w, n) for w, n in freq_saij.most_common() if w not in freq_piloto]
    terms = ["camara", "casacion", "sumario", "recurso", "sentencia", "amparo", "despido", "patrono", "empleador", "trabajador", "aguinaldo", "indemnizacion"]
    rows = []
    saij_norm = train.texto.map(sin_tildes)
    pilot_norm = piloto.texto.map(sin_tildes)
    for term in terms:
        pattern = r"\b" + re.escape(term) + r"\b"
        rows.append(dict(termino_predefinido=term,
                         saij_textos_pct=100 * saij_norm.str.contains(pattern, regex=True).mean(),
                         piloto_textos_pct=100 * pilot_norm.str.contains(pattern, regex=True).mean()))
    display(pd.DataFrame(rows).set_index("termino_predefinido").round(1))
    print("Lista jurídica predefinida: no etiqueta automáticamente términos como exclusivos de un país.")
else:
    marcar("Brecha de vocabulario", "PENDIENTE: requiere train y piloto revisado activado")


Brecha de vocabulario: PENDIENTE: requiere train y piloto revisado activado


## 7. Cierre del checkpoint
Completar tras ejecutar con datos reales:

- [ ] Registrar conteos, longitudes y cobertura de metadatos observados.
- [ ] Explicar `familia` como etiqueta débil y composición observada de `otros`.
- [ ] Interpretar duplicados, señales superficiales y límites del control por similitud.
- [ ] Comparar el piloto revisado sin exponer sus textos ni vocabulario identificable.
- [ ] Leer sección 8 de PLAN_DATOS_RAW.md cuando esté disponible.
- [ ] Confirmar licencias en sus fuentes; este notebook no las verifica.
- [ ] Integrar resultados comprobados en la sección 4.2.
- [ ] Borrar todas las salidas antes de versionar el notebook.

**Advertencia documental:** `src/data/saij.py` contiene una descripción que dice
que los datos externos solo van a entrenamiento, mientras que `make_splits.py`
y las tareas describen el uso de SAIJ como fuente de todas las particiones.
No copiar esa frase sin corregir la explicación del informe. El README también
conserva referencias a resultados pendientes del bufete que difieren de las tareas.
No se modifican esos archivos de Nes en este avance.


In [8]:
display(pd.DataFrame(estado))
print("Este notebook no guarda resultados en disco ni declara CP2 completado automáticamente.")


,seccion,estado
0,train,PENDIENTE: archivo no disponible
1,val,PENDIENTE: archivo no disponible
2,Balance,PENDIENTE: faltan datos
3,Etiquetas,PENDIENTE: faltan train/val
4,Piloto,DESACTIVADO: requiere revisión humana local
5,Longitudes,PENDIENTE: faltan datos
6,Metadatos de split,PENDIENTE: falta splits_info.json; referencia ...
7,Duplicados cruzados,PENDIENTE: se necesitan train y val
8,Atajos,PENDIENTE: faltan textos
9,Brecha de vocabulario,PENDIENTE: requiere train y piloto revisado ac...


Este notebook no guarda resultados en disco ni declara CP2 completado automáticamente.
